# M3 — Module E: CPU pipeline và export dataset
Chọn CPU. Mount Drive bằng giao diện Colab trước nếu dùng đường dẫn Drive.
Notebook clone commit đã được bạn push; điền COMMIT_SHA đầy đủ. Chế độ artifacts
kiểm tra labels/scores có sẵn; full chỉ dùng khi A–D đã được bàn giao.
Notebook không upload HF. Input dùng chung cả nhóm, OWNER chỉ chọn output cá nhân.


In [ ]:
from pathlib import Path
import json, re, subprocess, sys, uuid
from datetime import datetime, timezone

OWNER = 'trong'
MODE = 'artifacts'  # 'artifacts' hoặc 'full'
COMMIT_SHA = ''     # Điền SHA 40 ký tự của commit đã push có code M3
REPO_URL = 'https://github.com/ChickenDev-cyber/arxiv-data-science.git'
REPO_DIR = Path('/content/arxiv-m3')
DRIVE_ROOT = Path('/content/drive/MyDrive/arXiv-data/milestone 3')
INPUT_ROOT = DRIVE_ROOT / 'shared-inputs'
INVENTORY = INPUT_ROOT / 'inventory.json'
MANUAL_PAPERS = INPUT_ROOT / 'manual_papers.parquet'
LABELS = INPUT_ROOT / 'labels.parquet'
SCORES = INPUT_ROOT / 'scores-model.parquet'
MANUAL_DIR = INPUT_ROOT / 'manual'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8]
OUTPUT_DIR = DRIVE_ROOT / 'runs' / OWNER / RUN_ID
assert OWNER in ('trong','huynh-nhi','man-nhi','khang','hoang')
assert MODE in ('artifacts','full')
assert re.fullmatch(r'[0-9a-fA-F]{40}', COMMIT_SHA), 'Hãy điền COMMIT_SHA đã push trước khi clone.'
assert Path('/content/drive/MyDrive').is_dir(), 'Hãy mount Drive trước bằng giao diện Colab.'
print('Output:', OUTPUT_DIR)


In [ ]:
if not REPO_DIR.exists():
    subprocess.run(['git','clone','--no-checkout',REPO_URL,str(REPO_DIR)], check=True)
else:
    assert not subprocess.check_output(['git','-C',str(REPO_DIR),'status','--porcelain'], text=True).strip(), 'Repo local có thay đổi; chọn REPO_DIR mới.'
subprocess.run(['git','-C',str(REPO_DIR),'fetch','origin',COMMIT_SHA], check=True)
subprocess.run(['git','-C',str(REPO_DIR),'checkout','--detach',COMMIT_SHA], check=True)
subprocess.run([sys.executable,'-m','pip','install','-r',str(REPO_DIR/'src/requirements-m3.txt')], check=True)
sys.path.insert(0,str(REPO_DIR/'src'))
print('Commit:', subprocess.check_output(['git','-C',str(REPO_DIR),'rev-parse','HEAD'], text=True).strip())


In [ ]:
config = dict(mode=MODE, inventory=str(INVENTORY), manual_papers=str(MANUAL_PAPERS),
              output_dir=str(OUTPUT_DIR), owners=[OWNER], seed=42)
if MODE == 'artifacts':
    config.update(labels=str(LABELS), scores=str(SCORES))
else:
    config.update(manual_dir=str(MANUAL_DIR))
CONFIG_PATH = Path('/content') / f'm3-{OWNER}-{RUN_ID}.json'
CONFIG_PATH.write_text(json.dumps(config, indent=2))
from arxiv_pipeline.matching.pipeline import preflight
preflight(CONFIG_PATH)
print('Preflight OK; chưa ghi output run.')


In [ ]:
from arxiv_pipeline.matching.pipeline import run_pipeline
RUN_DIR = run_pipeline(CONFIG_PATH)
state = json.loads((RUN_DIR/'run_config.json').read_text())
print('Status:', state['status'], '| Evaluation:', state['evaluation_status'])
print('Dataset trên Drive:', RUN_DIR/'export'/OWNER)


In [ ]:
import pyarrow.parquet as pq
DATASET = RUN_DIR / 'export' / OWNER
for table in ('papers','references','elements','hierarchy','bibtex','predictions'):
    files = sorted((DATASET/'data'/table).glob('*.parquet'))
    print(table, 'shards=', len(files), 'rows=', sum(pq.ParquetFile(p).metadata.num_rows for p in files))
print('Dataset card:', DATASET/'README.md')


## Sau khi chạy
- Artifacts mode chưa huấn luyện/đánh giá MRR; xem trạng thái trong run_config.json.
- Nếu lỗi, đọc failed_stage và error; dùng RUN_ID mới cho lần chạy tiếp theo.
- Đối chiếu validator của Hoàng trước khi upload. Dataset cá nhân gồm README.md
  và data/6 bảng. Mỗi người tự upload HF, kiểm tra public/viewer/streaming và nộp link.
